Install Java & Download Hadoop




In [ ]:
# Install Java
!apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
!java -version

# Download Hadoop 3.3.6
!wget -q https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz
!mv hadoop-3.3.6 /usr/local/hadoop
print("Hadoop installed successfully")


Set Environment Variables

In [ ]:
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'
os.environ['HADOOP_HOME'] = '/usr/local/hadoop'
os.environ['PATH'] = os.environ['PATH'] + ':/usr/local/hadoop/bin:/usr/local/hadoop/sbin'
os.environ['HADOOP_CONF_DIR'] = '/usr/local/hadoop/etc/hadoop'

# Verify
!hadoop version


Configure Hadoop for Local Standalone Mode

In [ ]:
# In local standalone mode, NO XML config changes are needed.
# Hadoop runs as a single Java process using local filesystem (not HDFS).
# This is the simplest mode - perfect for testing on Colab.
print("Local standalone mode requires no extra configuration.")
print("Hadoop will use local filesystem directly.")


Create Input File

In [ ]:
!mkdir -p /content/wordcount_input

input_text = """Hadoop is a storage and processing tool.
Hadoop is a unit of MapReduce.
HDFS is a part of Hadoop.
MapReduce is a programming model for Big Data processing."""

with open('/content/wordcount_input/input.txt', 'w') as f:
    f.write(input_text)

print("Input file created:")
!cat /content/wordcount_input/input.txt


Write WC_Mapper.java

In [ ]:
mapper_code = '''package com.javatpoint;

import java.io.IOException;
import java.util.StringTokenizer;
import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.LongWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapred.MapReduceBase;
import org.apache.hadoop.mapred.Mapper;
import org.apache.hadoop.mapred.OutputCollector;
import org.apache.hadoop.mapred.Reporter;

public class WC_Mapper extends MapReduceBase
        implements Mapper<LongWritable, Text, Text, IntWritable> {

    private final static IntWritable one = new IntWritable(1);
    private Text word = new Text();

    public void map(LongWritable key, Text value,
            OutputCollector<Text, IntWritable> output,
            Reporter reporter) throws IOException {
        String line = value.toString();
        StringTokenizer tokenizer = new StringTokenizer(line);
        while (tokenizer.hasMoreTokens()) {
            word.set(tokenizer.nextToken());
            output.collect(word, one);
        }
    }
}
'''

!mkdir -p /content/wordcount/com/javatpoint
with open('/content/wordcount/WC_Mapper.java', 'w') as f:
    f.write(mapper_code)
print("WC_Mapper.java written.")


Write WC_Reducer.java

In [ ]:
reducer_code = '''package com.javatpoint;

import java.io.IOException;
import java.util.Iterator;
import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapred.MapReduceBase;
import org.apache.hadoop.mapred.OutputCollector;
import org.apache.hadoop.mapred.Reducer;
import org.apache.hadoop.mapred.Reporter;

public class WC_Reducer extends MapReduceBase
        implements Reducer<Text, IntWritable, Text, IntWritable> {

    public void reduce(Text key, Iterator<IntWritable> values,
            OutputCollector<Text, IntWritable> output,
            Reporter reporter) throws IOException {
        int sum = 0;
        while (values.hasNext()) {
            sum += values.next().get();
        }
        output.collect(key, new IntWritable(sum));
    }
}
'''

with open('/content/wordcount/WC_Reducer.java', 'w') as f:
    f.write(reducer_code)
print("WC_Reducer.java written.")


Write WC_Runner.java

In [ ]:
runner_code = '''package com.javatpoint;

import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapred.FileInputFormat;
import org.apache.hadoop.mapred.FileOutputFormat;
import org.apache.hadoop.mapred.JobClient;
import org.apache.hadoop.mapred.JobConf;
import org.apache.hadoop.mapred.TextInputFormat;
import org.apache.hadoop.mapred.TextOutputFormat;

public class WC_Runner {
    public static void main(String[] args) throws Exception {
        JobConf conf = new JobConf(WC_Runner.class);
        conf.setJobName("WordCount");

        conf.setOutputKeyClass(Text.class);
        conf.setOutputValueClass(IntWritable.class);

        conf.setMapperClass(WC_Mapper.class);
        conf.setReducerClass(WC_Reducer.class);

        conf.setInputFormat(TextInputFormat.class);
        conf.setOutputFormat(TextOutputFormat.class);

        FileInputFormat.setInputPaths(conf, new Path(args[0]));
        FileOutputFormat.setOutputPath(conf, new Path(args[1]));

        JobClient.runJob(conf);
    }
}
'''

with open('/content/wordcount/WC_Runner.java', 'w') as f:
    f.write(runner_code)
print("WC_Runner.java written.")


Compile All Java Files

In [ ]:
!javac -classpath $(hadoop classpath) \
       -source 8 -target 8 \
       -d /content/wordcount \
       /content/wordcount/WC_Mapper.java \
       /content/wordcount/WC_Reducer.java \
       /content/wordcount/WC_Runner.java

print("Compilation done.")


Create JAR File

In [ ]:
import os
os.chdir('/content/wordcount')
!jar -cvf wordcount.jar -C /content/wordcount .
print("JAR created.")


Run the MapReduce Job

In [ ]:
# Remove output dir if it exists from a previous run
!rm -rf /content/wordcount_output

!hadoop jar /content/wordcount/wordcount.jar \
            com.javatpoint.WC_Runner \
            /content/wordcount_input \
            /content/wordcount_output 2>&1

print("MapReduce job completed.")


View Output

In [ ]:
print("===== WORDCOUNT OUTPUT =====")
!cat /content/wordcount_output/*


In [ ]:
# ============================================================
# IMPORTANT NOTES FOR ORAL EXAM
# WORDCOUNT APPLICATION USING MapReduce (Google Colab)
# ============================================================

# ------------------------------------------------------------
# WHAT IS THIS PRACTICAL ABOUT?
# ------------------------------------------------------------
# This practical counts how many times each word appears in a
# text file using Hadoop MapReduce on Google Colab.
# It runs in LOCAL STANDALONE MODE - no HDFS, no cluster needed.
# Hadoop runs as a single Java process on the local filesystem.
#
# Three Java files used:
#   WC_Mapper.java  - The Mapper class
#   WC_Reducer.java - The Reducer class
#   WC_Runner.java  - The Driver/Runner class

# ------------------------------------------------------------
# WHAT IS HADOOP?
# ------------------------------------------------------------
# Hadoop is an open-source framework for storing and processing
# large amounts of data (Big Data) across a cluster of computers.
# Two main components:
#   1. HDFS (Hadoop Distributed File System) - for STORAGE
#   2. MapReduce                             - for PROCESSING

# ------------------------------------------------------------
# WHAT IS LOCAL STANDALONE MODE?
# ------------------------------------------------------------
# Hadoop has 3 modes:
#   1. Local Standalone Mode  - Single Java process, local filesystem.
#                               No HDFS, no daemons. Used for testing.
#   2. Pseudo-Distributed Mode- All daemons run on one machine.
#                               Uses HDFS on local machine.
#   3. Fully Distributed Mode - Multiple machines in a cluster.
#                               Real production setup.
#
# In this practical we use LOCAL STANDALONE MODE because:
#   - Google Colab is a single machine
#   - No need to start NameNode, DataNode, ResourceManager etc.
#   - Input/output are local paths (/content/...) not HDFS paths (/input)
#   - No start-dfs.sh or start-yarn.sh needed

# ------------------------------------------------------------
# WHAT IS MapReduce?
# ------------------------------------------------------------
# MapReduce is a programming model for processing large datasets
# in parallel. THREE phases:
#
#   1. MAP PHASE:
#      - Mapper reads (key, value) pairs and emits intermediate pairs.
#      - WordCount: reads (line_offset, line_text), emits (word, 1).
#
#   2. SHUFFLE AND SORT (automatic, done by Hadoop):
#      - Groups all values for the same key together.
#      - All (word, 1) pairs with the same word go to same Reducer.
#      - Sorted alphabetically by key.
#
#   3. REDUCE PHASE:
#      - Reducer receives (key, list_of_values), produces final output.
#      - WordCount: receives (word, [1,1,1,...]), sums -> (word, count).

# ------------------------------------------------------------
# CODE EXPLANATION - WC_Mapper
# ------------------------------------------------------------
# implements Mapper<LongWritable, Text, Text, IntWritable>
#   - Four type parameters: <InputKey, InputValue, OutputKey, OutputValue>
#   - Input Key   : LongWritable - byte offset of the line in the file
#   - Input Value : Text         - one full line of text
#   - Output Key  : Text         - a single word
#   - Output Value: IntWritable  - always 1 (one occurrence)
#
# private final static IntWritable one = new IntWritable(1);
#   - Constant value 1 shared across all map() calls.
#   - static + final = avoid creating new object every time -> efficient.
#
# private Text word = new Text();
#   - Reusable Text object to hold each word -> memory efficient.
#
# map() method step by step:
#   String line = value.toString();
#     -> Converts Hadoop Text to regular Java String.
#
#   StringTokenizer tokenizer = new StringTokenizer(line);
#     -> Splits the line into words by whitespace (space, tab, newline).
#
#   while (tokenizer.hasMoreTokens()) {
#     -> Loops through each word in the line.
#
#     word.set(tokenizer.nextToken());
#     -> Gets next word, stores in reusable Text object.
#
#     output.collect(word, one);
#     -> Emits (word, 1) as intermediate key-value pair.
#   }

# ------------------------------------------------------------
# CODE EXPLANATION - WC_Reducer
# ------------------------------------------------------------
# implements Reducer<Text, IntWritable, Text, IntWritable>
#   - Receives: (word, [1, 1, 1, ...]) - all 1s for the same word
#   - Sums up all the 1s
#   - Emits: (word, total_count)
#   - Example: ("Hadoop", [1, 1]) -> ("Hadoop", 2)
#
# reduce() method step by step:
#   int sum = 0;
#     -> Initialize counter to 0.
#
#   while (values.hasNext()) { sum += values.next().get(); }
#     -> Loop through all 1s and add them up.
#
#   output.collect(key, new IntWritable(sum));
#     -> Emit (word, total_count) as final output.

# ------------------------------------------------------------
# CODE EXPLANATION - WC_Runner
# ------------------------------------------------------------
# JobConf conf = new JobConf(WC_Runner.class);
#   -> Creates job configuration object.
#
# conf.setJobName("WordCount");
#   -> Sets a name for the job (visible in logs).
#
# conf.setMapperClass(WC_Mapper.class);
# conf.setReducerClass(WC_Reducer.class);
#   -> Tells Hadoop which Mapper and Reducer to use.
#
# conf.setOutputKeyClass(Text.class);
# conf.setOutputValueClass(IntWritable.class);
#   -> Defines the data types of the final output.
#
# FileInputFormat.setInputPaths(conf, new Path(args[0]));
# FileOutputFormat.setOutputPath(conf, new Path(args[1]));
#   -> Sets input and output paths passed as command-line arguments.
#
# JobClient.runJob(conf);
#   -> Submits and runs the job.

# ------------------------------------------------------------
# FULL MapReduce FLOW FOR WORDCOUNT
# ------------------------------------------------------------
# INPUT:
#   "Hadoop is a storage and processing tool."
#   "Hadoop is a unit of MapReduce."
#
# AFTER MAP:
#   (Hadoop,1), (is,1), (a,1), (storage,1), (processing,1), (tool.,1)
#   (Hadoop,1), (is,1), (a,1), (unit,1), (of,1), (MapReduce.,1)
#
# AFTER SHUFFLE & SORT (automatic):
#   (Hadoop,    [1, 1])
#   (MapReduce.,[1])
#   (a,         [1, 1])
#   (is,        [1, 1])
#   ...
#
# AFTER REDUCE:
#   Hadoop      2
#   MapReduce.  1
#   a           2
#   is          2
#   ...

# ------------------------------------------------------------
# KEY CLASSES USED
# ------------------------------------------------------------
# LongWritable   : Hadoop serializable long. Used for line byte offset.
# IntWritable    : Hadoop serializable int.  Used for count = 1.
# Text           : Hadoop serializable String. Used for words.
# MapReduceBase  : Base class providing default implementations.
# Mapper         : Interface defining the map() method signature.
# Reducer        : Interface defining the reduce() method signature.
# OutputCollector: Collects (key, value) output from Mapper/Reducer.
# Reporter       : Reports job progress to Hadoop framework.
# StringTokenizer: Java class to split a string into words by whitespace.
# JobConf        : Holds all configuration for a MapReduce job.
# JobClient      : Submits the job to Hadoop for execution.
#
# WHY Hadoop types instead of Java types?
#   Hadoop needs to SERIALIZE data to send it across the network.
#   Hadoop types implement Writable interface for serialization.
#   Regular Java int/String cannot be directly serialized by Hadoop.

# ------------------------------------------------------------
# WHY -source 8 -target 8 IN COMPILATION?
# ------------------------------------------------------------
# Google Colab has Java 17 installed by default.
# Hadoop 3.3.6 on Colab runs using Java 8 (class file version 52.0).
# If compiled with Java 17, class file version = 61.0 -> ERROR.
# -source 8 -target 8 forces javac to produce Java 8 bytecode.
# This makes the compiled classes compatible with Hadoop's Java 8 runtime.

# ------------------------------------------------------------
# COLAB vs UBUNTU - KEY DIFFERENCES
# ------------------------------------------------------------
# Ubuntu (HDFS mode):                  Google Colab (Standalone mode):
#   start-dfs.sh / start-yarn.sh         No daemons needed
#   hdfs dfs -mkdir /input               mkdir /content/wordcount_input
#   hdfs dfs -put input.txt /input        write file directly with Python
#   hadoop jar ... /input /r_output       hadoop jar ... /content/input /content/output
#   hdfs dfs -cat /r_output/part-00000    cat /content/wordcount_output/*
#   jps shows 5 daemons                  jps shows nothing extra

# ------------------------------------------------------------
# KEY COMMANDS USED IN THIS NOTEBOOK
# ------------------------------------------------------------
# apt-get install openjdk-8-jdk-headless  : Install Java 8
# wget <url>                              : Download Hadoop tar file
# tar -xzf hadoop-3.3.6.tar.gz           : Extract Hadoop
# os.environ['JAVA_HOME'] = ...          : Set Java path for Hadoop
# os.environ['HADOOP_HOME'] = ...        : Set Hadoop installation path
# hadoop version                         : Verify Hadoop is working
# javac -classpath $(hadoop classpath)   : Compile with Hadoop JARs
# -source 8 -target 8                    : Compile targeting Java 8
# jar -cvf wordcount.jar -C . .          : Package classes into JAR
# hadoop jar wordcount.jar WC_Runner ... : Run the MapReduce job
# cat /content/wordcount_output/*        : View output (any filename)

# ------------------------------------------------------------
# POSSIBLE ORAL EXAM QUESTIONS AND ANSWERS
# ------------------------------------------------------------
# Q: What is Hadoop?
# A: Hadoop is an open-source framework for distributed storage and
#    processing of large datasets (Big Data).
#    Two main components: HDFS (storage) and MapReduce (processing).

# Q: What is MapReduce?
# A: A programming model for processing large data in parallel.
#    Map phase emits (key, value) pairs.
#    Shuffle & Sort groups values by key automatically.
#    Reduce phase aggregates values to produce final output.

# Q: What are the 3 modes of Hadoop?
# A: 1. Local Standalone - single process, local filesystem, no daemons.
#    2. Pseudo-Distributed - all daemons on one machine, uses HDFS.
#    3. Fully Distributed - multiple machines, real cluster setup.

# Q: Why do we use Local Standalone mode in Colab?
# A: Colab is a single machine environment. Local standalone mode
#    requires no HDFS setup, no daemons, and uses the local filesystem
#    directly. It is the simplest way to test MapReduce programs.

# Q: What does the Mapper do in WordCount?
# A: Reads each line, splits it into words using StringTokenizer,
#    and emits (word, 1) for every word found.

# Q: What does the Reducer do in WordCount?
# A: Receives (word, [1,1,1,...]) for each word, sums all the 1s,
#    and emits (word, total_count) as the final output.

# Q: What is Shuffle and Sort?
# A: The automatic phase between Map and Reduce.
#    All intermediate (key, value) pairs are sorted by key and
#    all values for the same key are grouped and sent to the same Reducer.

# Q: Why do we use IntWritable instead of int?
# A: Hadoop needs to serialize data to send it across the network.
#    IntWritable implements the Writable interface for serialization.
#    Regular Java int cannot be directly serialized by Hadoop.

# Q: What is StringTokenizer?
# A: A Java class that splits a string into tokens (words) by whitespace.
#    Used in WordCount to split each line into individual words.

# Q: What is the purpose of the Runner/Driver class?
# A: Configures and submits the MapReduce job.
#    Sets input/output paths, Mapper class, Reducer class,
#    output key/value types, then runs the job.

# Q: Why do we use -source 8 -target 8 in javac?
# A: Colab has Java 17 but Hadoop runs on Java 8.
#    Without this flag, compiled bytecode is version 61 (Java 17)
#    which Hadoop cannot load (supports only up to version 52 = Java 8).
#    These flags force compilation to Java 8 compatible bytecode.

# Q: What is output.collect()?
# A: Method of OutputCollector used in old Hadoop API (mapred package).
#    It sends a (key, value) pair from the Mapper or Reducer
#    to the next stage of the MapReduce pipeline.

# Q: What is the difference between mapred and mapreduce packages?
# A: mapred   = old Hadoop API. Uses OutputCollector, Reporter, Iterator.
#    mapreduce = new Hadoop API. Uses Context object for writing output.
#    Both work but new API (mapreduce) is preferred for Hadoop 3.x.

# Q: What is a JAR file and why do we create one?
# A: JAR = Java Archive. Packages all compiled .class files together.
#    Hadoop needs a JAR to distribute and run the job across nodes.
#    Created using: jar -cvf wordcount.jar -C . .

# Q: What is the expected output of WordCount?
# A: Each unique word followed by its count, sorted alphabetically.
#    Example: Big=1, Data=1, HDFS=1, Hadoop=3, MapReduce=2, a=3, is=3...

# Q: What does Map input records=4 mean in the logs?
# A: The Mapper processed 4 lines of input (our input file has 4 lines).

# Q: What does Map output records=28 mean in the logs?
# A: The Mapper emitted 28 (word, 1) pairs total across all 4 lines.

# Q: What does Reduce input groups=20 mean in the logs?
# A: There are 20 unique words in the input. Each group goes to Reducer.

# Q: What does Reduce output records=20 mean in the logs?
# A: The Reducer produced 20 final (word, count) output pairs.

# ============================================================
print("Oral exam notes loaded successfully.")
